# ETL Pipeline - Data Preprocessing, Transformation, and Loading
## CODTECH Internship - Task 1

**Objective:** Create a complete Data Pipeline for preprocessing, transformation, and loading using Pandas and Scikit-Learn.

**What is an ETL Pipeline?**
- **Extract:** Load data from source
- **Transform:** Clean, preprocess, and feature engineer the data
- **Load:** Save processed data to destination

This notebook demonstrates a professional-grade ETL pipeline with detailed explanations and outputs.

## Section 1: Import Required Libraries

In this section, we import all the necessary libraries for data processing and analysis.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
import warnings
import os
from datetime import datetime

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Section 2: Load Data from Source

Load raw data from CSV file and examine its structure, data types, and basic statistics.

In [ ]:
# Load data from CSV file
input_file = 'data/sample_data.csv'

# Check if file exists
if os.path.exists(input_file):
    df_raw = pd.read_csv(input_file)
    print("✓ Data loaded successfully!")
    print(f"\nDataset Shape: {df_raw.shape}")
    print(f"  - Rows: {df_raw.shape[0]}")
    print(f"  - Columns: {df_raw.shape[1]}")
else:
    print(f"✗ File not found: {input_file}")

# Display first few rows
print("\n📊 First 10 rows:")
print(df_raw.head(10))

In [ ]:
# Display data types
print("\n📌 Data Types:")
print(df_raw.dtypes)

# Display basic statistics
print("\n📈 Statistical Summary:")
print(df_raw.describe())

## Section 3: Data Cleaning and Preprocessing

Handle missing values, remove duplicates, correct data types, and address data quality issues.

In [ ]:
# Create a copy of the raw data for preprocessing
df_cleaned = df_raw.copy()

# Step 1: Check and handle missing values
print("="*60)
print("STEP 1: HANDLING MISSING VALUES")
print("="*60)

missing_data = df_cleaned.isnull().sum()
print("\nMissing Values Count:")
print(missing_data[missing_data > 0])

# Separate numeric and categorical columns
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")

# Impute missing values - Numeric columns with mean
if len(numeric_cols) > 0:
    numeric_imputer = SimpleImputer(strategy='mean')
    df_cleaned[numeric_cols] = numeric_imputer.fit_transform(df_cleaned[numeric_cols])
    print(f"\n✓ Applied mean imputation to {len(numeric_cols)} numeric columns")

# Impute missing values - Categorical columns with mode
if len(categorical_cols) > 0:
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    df_cleaned[categorical_cols] = categorical_imputer.fit_transform(df_cleaned[categorical_cols])
    print(f"✓ Applied mode imputation to {len(categorical_cols)} categorical columns")

# Verify no missing values remain
print(f"\nMissing values after imputation: {df_cleaned.isnull().sum().sum()}")

In [ ]:
# Step 2: Remove duplicate records
print("\n" + "="*60)
print("STEP 2: REMOVING DUPLICATES")
print("="*60)

initial_rows = len(df_cleaned)
df_cleaned = df_cleaned.drop_duplicates()
duplicates_removed = initial_rows - len(df_cleaned)

print(f"Initial rows: {initial_rows}")
print(f"After removing duplicates: {len(df_cleaned)}")
print(f"Duplicates removed: {duplicates_removed}")

# Step 3: Data quality summary
print("\n" + "="*60)
print("STEP 3: DATA QUALITY SUMMARY")
print("="*60)

print(f"\nDataset shape: {df_cleaned.shape}")
print(f"Missing values: {df_cleaned.isnull().sum().sum()}")
print(f"Duplicate rows: {df_cleaned.duplicated().sum()}")
print(f"Memory usage: {df_cleaned.memory_usage(deep=True).sum() / 1024:.2f} KB")

print("\n✓ Data cleaning completed successfully!")

## Section 4: Data Transformation

Apply transformations such as encoding categorical variables, standardization, and feature engineering.

In [ ]:
# Create a copy for transformation
df_transformed = df_cleaned.copy()

# Define column types
numeric_cols = df_transformed.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_transformed.select_dtypes(include=['object']).columns.tolist()

print("="*60)
print("TRANSFORMATION STEP 1: ENCODING CATEGORICAL VARIABLES")
print("="*60)

label_encoders = {}

# Encode categorical variables (convert text to numbers)
for col in categorical_cols:
    le = LabelEncoder()
    df_transformed[col] = le.fit_transform(df_transformed[col].astype(str))
    label_encoders[col] = le
    print(f"✓ Encoded '{col}' with {len(le.classes_)} unique values")
    print(f"  Classes: {le.classes_}")

print("\n" + "="*60)
print("TRANSFORMATION STEP 2: STANDARDIZATION (SCALING)")
print("="*60)

# Standardize numerical features
scaler = StandardScaler()
df_transformed[numeric_cols] = scaler.fit_transform(df_transformed[numeric_cols])

print(f"✓ Standardized {len(numeric_cols)} numerical columns")
print(f"  Mean: ~0, Standard Deviation: ~1")
print(f"  This makes features comparable on the same scale")

# Display statistics after scaling
print("\nScaled data statistics:")
print(df_transformed[numeric_cols].describe())

In [ ]:
# Feature Engineering - Create derived features
print("\n" + "="*60)
print("TRANSFORMATION STEP 3: FEATURE ENGINEERING")
print("="*60)

# Example: Create interaction features
if len(numeric_cols) >= 2:
    col1, col2 = numeric_cols[0], numeric_cols[1]
    df_transformed[f'{col1}_to_{col2}_ratio'] = df_transformed[col1] / (df_transformed[col2] + 1e-8)
    print(f"✓ Created derived feature: '{col1}_to_{col2}_ratio'")

print(f"\nFinal transformed dataset shape: {df_transformed.shape}")
print(f"Column names: {df_transformed.columns.tolist()}")

## Section 5: Data Validation

Validate the transformed data for quality, consistency, and completeness before loading.

In [ ]:
print("="*60)
print("DATA VALIDATION")
print("="*60)

# Validation checks
checks = {
    "Total Rows": len(df_transformed),
    "Total Columns": len(df_transformed.columns),
    "Missing Values": df_transformed.isnull().sum().sum(),
    "Duplicate Rows": df_transformed.duplicated().sum(),
    "Memory Usage (KB)": round(df_transformed.memory_usage(deep=True).sum() / 1024, 2)
}

for check_name, check_value in checks.items():
    status = "✓" if (check_value == 0 if isinstance(check_value, int)) or check_value > 0 else "✗"
    print(f"{status} {check_name}: {check_value}")

# Data type validation
print("\nData Types after Transformation:")
print(df_transformed.dtypes)

# Display sample of transformed data
print("\nSample of Transformed Data (first 5 rows):")
print(df_transformed.head())

print("\n✓ Data validation completed successfully!")

## Section 6: Load Transformed Data

Save the processed data to CSV file for use in machine learning models or analysis.

In [ ]:
print("="*60)
print("LOADING TRANSFORMED DATA")
print("="*60)

# Create output directory if it doesn't exist
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

# Define output file path
output_file = os.path.join(output_dir, 'processed_data.csv')

# Save transformed data to CSV
df_transformed.to_csv(output_file, index=False)

# Get file information
file_size = os.path.getsize(output_file) / 1024  # Size in KB

print(f"\n✓ Data saved successfully!")
print(f"  Output file: {output_file}")
print(f"  File size: {file_size:.2f} KB")
print(f"  Records saved: {len(df_transformed)}")
print(f"  Columns saved: {len(df_transformed.columns)}")

# Summary comparison
print("\n" + "="*60)
print("ETL PIPELINE SUMMARY")
print("="*60)
print(f"\nInput Data:")
print(f"  - Original shape: {df_raw.shape}")
print(f"  - Missing values: {df_raw.isnull().sum().sum()}")

print(f"\nOutput Data:")
print(f"  - Final shape: {df_transformed.shape}")
print(f"  - Missing values: {df_transformed.isnull().sum().sum()}")
print(f"  - Data quality: 100%")

print(f"\n✓ ETL Pipeline Completed Successfully!")
print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Bonus: Data Quality Comparison

Let's create a visual comparison of the data before and after the ETL pipeline.

In [ ]:
# Create comparison table
comparison_data = {
    'Metric': ['Total Rows', 'Total Columns', 'Missing Values', 'Duplicate Rows', 'Encoded Columns', 'Scaled Columns'],
    'Raw Data': [
        df_raw.shape[0],
        df_raw.shape[1],
        df_raw.isnull().sum().sum(),
        df_raw.duplicated().sum(),
        0,
        0
    ],
    'Processed Data': [
        df_transformed.shape[0],
        df_transformed.shape[1],
        df_transformed.isnull().sum().sum(),
        df_transformed.duplicated().sum(),
        len(categorical_cols),
        len(numeric_cols)
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n📊 ETL PIPELINE COMPARISON TABLE")
print("="*60)
print(comparison_df.to_string(index=False))

print("\n✨ Key Improvements:")
print(f"  ✓ All missing values handled ({df_raw.isnull().sum().sum()} → {df_transformed.isnull().sum().sum()})")
print(f"  ✓ All categorical data encoded ({len(categorical_cols)} columns)")
print(f"  ✓ All numerical data standardized ({len(numeric_cols)} columns)")
print(f"  ✓ New features engineered (1 derived feature)")
print(f"  ✓ Data ready for machine learning!")